In [1]:
import cv2
import mediapipe as mp
import numpy as np
import os
import glob

# ==========================================
# CONFIGURATION
# ==========================================
DATA_DIR = "gesture_videos"  # Folder containing your recorded videos

# 1. MEDIAPIPE SETUP
mp_holistic = mp.solutions.holistic

def landmark_xy(landmarks, idx):
    lm = landmarks[idx]
    return lm.x, lm.y

def center(a, b):
    return ((a[0] + b[0]) * 0.5, (a[1] + b[1]) * 0.5)

def analyze_videos():
    print("=== STARTING CALIBRATION ===")
    print(f"Scanning folder: {DATA_DIR}\n")

    # Storage for collected metrics
    lean_values_left = []
    lean_values_right = []
    torso_ratios_duck = []
    wrist_heights_jump = []

    # Find all videos
    video_files = []
    for ext in ['*.mp4', '*.avi', '*.mov']:
        video_files.extend(glob.glob(os.path.join(DATA_DIR, '**', ext), recursive=True))

    with mp_holistic.Holistic(static_image_mode=False, model_complexity=1) as holistic:
        
        for video_path in video_files:
            filename = os.path.basename(video_path).lower()
            gesture_type = "neutral"
            
            # Identify gesture from filename
            if "left" in filename: gesture_type = "left"
            elif "right" in filename: gesture_type = "right"
            elif "duck" in filename: gesture_type = "duck"
            elif "jump" in filename: gesture_type = "jump"
            else: continue # Skip neutral or unknown

            cap = cv2.VideoCapture(video_path)
            
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret: break

                # Process Frame
                image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = holistic.process(image)
                
                if not results.pose_landmarks: continue
                lms = results.pose_landmarks.landmark

                # --- CALCULATE METRICS ---
                
                # 1. LEAN (Left/Right)
                # Calculate shoulder center and hip center
                lsho = landmark_xy(lms, mp_holistic.PoseLandmark.LEFT_SHOULDER)
                rsho = landmark_xy(lms, mp_holistic.PoseLandmark.RIGHT_SHOULDER)
                lhip = landmark_xy(lms, mp_holistic.PoseLandmark.LEFT_HIP)
                rhip = landmark_xy(lms, mp_holistic.PoseLandmark.RIGHT_HIP)
                
                sh_cx, _ = center(lsho, rsho)
                hip_cx, _ = center(lhip, rhip)
                dx = sh_cx - hip_cx # Negative = Left, Positive = Right

                if gesture_type == "left": lean_values_left.append(dx)
                if gesture_type == "right": lean_values_right.append(dx)

                # 2. DUCK (Torso Ratio)
                nose = landmark_xy(lms, mp_holistic.PoseLandmark.NOSE)
                _, hip_cy = center(lhip, rhip)
                torso_len = abs(nose[1] - hip_cy)
                
                if gesture_type == "duck": torso_ratios_duck.append(torso_len)

                # 3. JUMP (Wrist vs Shoulder)
                lwri = landmark_xy(lms, mp_holistic.PoseLandmark.LEFT_WRIST)
                rwri = landmark_xy(lms, mp_holistic.PoseLandmark.RIGHT_WRIST)
                shoulder_y = (lsho[1] + rsho[1]) * 0.5
                
                # How far ABOVE shoulder are wrists? (Note: Y decreases going up)
                # If wrist is above shoulder, value is positive
                l_dist = shoulder_y - lwri[1]
                r_dist = shoulder_y - rwri[1]
                
                if gesture_type == "jump": 
                    # Store the smaller of the two hands (both hands must be up)
                    wrist_heights_jump.append(min(l_dist, r_dist))

            cap.release()
            print(f"Processed {filename}...")

    # ==========================================
    # RESULTS & SUGGESTIONS
    # ==========================================
    print("\n" + "="*40)
    print("CALIBRATION RESULTS & SUGGESTIONS")
    print("="*40)

    # --- LEFT / RIGHT ---
    print(f"\n[LEAN THRESHOLD]")
    print(f"Current Setting: 0.06 (Example)")
    if lean_values_left and lean_values_right:
        # We take the 80th percentile to be safe (ignore extreme outliers)
        avg_left = np.percentile(np.abs(lean_values_left), 50) 
        avg_right = np.percentile(np.abs(lean_values_right), 50)
        suggested_lean = min(avg_left, avg_right) * 0.8 # Set threshold at 80% of average lean
        print(f"  - Avg Left Lean observed:  {avg_left:.4f}")
        print(f"  - Avg Right Lean observed: {avg_right:.4f}")
        print(f"  -> SUGGESTED NEW VALUE:    {suggested_lean:.4f}")
    else:
        print("  - No Left/Right data found.")

    # --- DUCK ---
    print(f"\n[CROUCH TORSO RATIO]")
    print(f"Current Setting: 0.62 (Example)")
    if torso_ratios_duck:
        # For ducking, we want the MAX torso length seen while ducking
        # But to distinguish from standing, we usually look for 'smaller than X'
        avg_duck_torso = np.mean(torso_ratios_duck)
        # Suggest a value slightly higher than the average duck height so it catches it
        suggested_duck = avg_duck_torso * 1.1 
        print(f"  - Avg Torso size while ducking: {avg_duck_torso:.4f}")
        print(f"  -> SUGGESTED NEW VALUE:         {suggested_duck:.4f}")
    else:
        print("  - No Duck data found.")

    # --- JUMP ---
    print(f"\n[HANDS ABOVE SHOULDER DELTA]")
    print(f"Current Setting: 0.02 (Example)")
    if wrist_heights_jump:
        # Average distance wrists were above shoulders
        avg_jump_height = np.mean(wrist_heights_jump)
        suggested_jump = avg_jump_height * 0.8
        print(f"  - Avg wrist height above shoulder: {avg_jump_height:.4f}")
        print(f"  -> SUGGESTED NEW VALUE:            {suggested_jump:.4f}")
    else:
        print("  - No Jump data found.")

if __name__ == "__main__":
    analyze_videos()

=== STARTING CALIBRATION ===
Scanning folder: gesture_videos



c:\Users\LOQ\OneDrive\Desktop\codes\Applied AI Engineering Lab\Body-Game-Gesture-Detection\AiLab\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processed duck_1.mp4...
Processed duck_2.mp4...
Processed duck_3.mp4...
Processed jump_1.mp4...
Processed jump_2.mp4...
Processed jump_3.mp4...
Processed left_1.mp4...
Processed left_2.mp4...
Processed left_3.mp4...
Processed right_1.mp4...
Processed right_2.mp4...
Processed right_3.mp4...

CALIBRATION RESULTS & SUGGESTIONS

[LEAN THRESHOLD]
Current Setting: 0.06 (Example)
  - Avg Left Lean observed:  0.0395
  - Avg Right Lean observed: 0.0703
  -> SUGGESTED NEW VALUE:    0.0316

[CROUCH TORSO RATIO]
Current Setting: 0.62 (Example)
  - Avg Torso size while ducking: 0.7796
  -> SUGGESTED NEW VALUE:         0.8575

[HANDS ABOVE SHOULDER DELTA]
Current Setting: 0.02 (Example)
  - Avg wrist height above shoulder: -0.5148
  -> SUGGESTED NEW VALUE:            -0.4119
